In [ ]:
import os
import re
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate

# 1. Load environment variables from .env
load_dotenv()

# Verify presence of required keys
required_keys = ["GEMINI_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"]
for key in required_keys:
    status = "Loaded" if os.getenv(key) else "MISSING"
    print(f"{key}: {status}")

GEMINI_API_KEY: Loaded
GROQ_API_KEY: Loaded
TAVILY_API_KEY: Loaded


# Document Loading & PDF Parsing

Concept: LangChain document loaders standardize raw data sources (PDFs, HTML, Markdown) into Document objects containing page_content (string) and metadata (dict). We use PyPDFLoader to parse local PDFs into clean string context.

In [2]:
def load_resume_text(file_path: str) -> str:
    """Parses a multi-page PDF resume or docx resume into a consolidated text string."""

    if file_path.endswith('.pdf'):
        from langchain_community.document_loaders import PyMuPDFLoader

        loader = PyMuPDFLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and white-space gaps
        resume_context = re.sub(r"\xa0", " ", raw_text)

        return resume_context
    elif file_path.endswith(('.doc', 'docx')):
        from langchain_community.document_loaders import UnstructuredWordDocumentLoader

        loader = UnstructuredWordDocumentLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and redundant whitespace
        resume_context = re.sub(r"\xa0", " ", raw_text)
        resume_context = re.sub(r"\n\s*\n", "\n\n", resume_context)

        return resume_context
    
    else:
        raise ValueError(
            f"Unsupported file extension for '{file_path}'. "
            "Only .pdf, .doc, and .docx formats are supported."
        )




In [3]:
# Example usage:
resume_context = load_resume_text("data/resume.pdf")

print(f"Parsed Resume Length: {len(resume_context)} characters")
print("-"*100)
print(resume_context)

/var/folders/q7/r62zxy213ms4wg9xz6cljcnm0000gn/T/ipykernel_31244/1162988709.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Parsed Resume Length: 6334 characters
----------------------------------------------------------------------------------------------------
Hamza Ahmed Siddiqui 
Data Scientist & ML Engineer · Computer Vision & NLP · AI Solutions 
hamzaahmedsiddiqui10@gmail.com | +49 1575 5794618 | Göttingen, Germany | LinkedIn | GitHub 
 
WoRk ExPERIEncE 
~Research Assistant - Computer Vision                                                                                           Apr 2026 – Present 
Eckerlab                                                                                                                                                Göttingen, Germany 
Project: Video Generalization Dataset 
•​ Investigated cross-domain generalization in video action recognition on Something-Something V2 (SSv2) by 
benchmarking feature representations from self-supervised vision models (DINOv2, V-JEPA Base/Giant) with 
trained attention heads.    
•​ Integrated Segment Anything Model (SAM 3) into the vi

# Job description 
Text file. different jobs separated by "------" line.


In [4]:
def load_and_clean_job_descriptions(file_path: str = "./job_desc.txt", remove_patterns: list = ['Easy Apply']) -> str:
    """Loads raw pasted LinkedIn job descriptions, splits by separator, and cleans whitespace."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_content = f.read()

    ## 1. major cleanup
    # Split jobs using the custom separator
    raw_jobs = raw_content.split("------")
    
    cleaned_jobs = []
    for i, job in enumerate(raw_jobs, 1):
        text = job.strip()
        if not text:
            continue
        
        # Collapse 3+ consecutive newlines down to 2 to trim unnecessary token bloat
        text = re.sub(r'\n\s*\n+', '\n\n', text)
        
        cleaned_jobs.append(f"====== JOB POSTING {i} ======\n{text}")
        
    formatted_job_data = "\n\n".join(cleaned_jobs)
    print(f"[*] Successfully processed {len(cleaned_jobs)} job posting(s). Total characters: {len(formatted_job_data)}")

    ## 2. Remove specific lines
    lines = formatted_job_data.splitlines()
    filtered_lines = [
            line for line in lines
            if not any(pattern.lower()==line.lower() for pattern in remove_patterns)
        ]

    text = "\n".join(filtered_lines)
    formatted_job_data = re.sub(r'\n\s*\n+', '\n\n', text)
     
    print(f"[*] Phase 2 Cleanup Complete. Total characters: {len(formatted_job_data)}")
    
    return formatted_job_data




# Quick test run:
blacklist = [
    "Save",
    "Apply",
    "Easy Apply",
    "No response insights available yet",
    "Your profile and resume match the required qualifications well",
    "See how you compare to over 100 other applicants",
    "See how you compare to over 100 others who clicked apply",
    "BETA • Is this information helpful?",
    "Restart Premium for €0",
    "Access exclusive applicant insights, see jobs where you have the highest chance of hearing back, and more.",
    "Show match details",
    "Responses managed off LinkedIn",
    "Message",
    ]

job_data = load_and_clean_job_descriptions("data/job_desc.txt", remove_patterns=blacklist)

print(job_data)

[*] Successfully processed 2 job posting(s). Total characters: 10074
[*] Phase 2 Cleanup Complete. Total characters: 9355
====== JOB POSTING 1 ======
AI Solutions Engineer 

Antwerp, Flemish Region, Belgium · 1 week ago · Over 100 applicants

Hybrid
Full-time

About the job
About Oper
Oper, a European Fintech company, is a leader in cloud-based digital mortgage software. We provide personalized journeys for borrowers and lenders. Oper's technology streamlines the mortgage application process, eliminates paperwork, and increases transparency.
The fastest growing part of our business is Herman, our AI agent for mortgage origination. Herman reads documents, checks them, and works a credit file the way an analyst would, inside the regulated workflow of a real bank. It runs on sovereign EU infrastructure, because our clients cannot send mortgage data anywhere else. Phase by phase, it automates more of the origination process: first employed borrowers, then the self-employed, and onward. Thi

# Pydantic Schema for Structured LLM Outputs
Concept: Standard LLMs return free-form text, which breaks downstream automation. Pydantic models define explicit Python data types and field descriptions. LangChain uses these definitions to force the model to respond strictly in a validated JSON schema.

In [5]:
from pydantic import BaseModel, Field
from typing import List

class JobTarget(BaseModel):
    company_name: str = Field(description="Name of the company hiring, or 'Unknown' if not mentioned in the posting.")
    job_title: str = Field(description="Target job title extracted directly from the posting.")

class ActionableRewrite(BaseModel):
    current_text: str = Field(description="Weak or generic phrasing currently found on the resume.")
    suggested_text: str = Field(description="Quantified, metric-driven rewrite tailored to real JD requirements.")
    reasoning: str = Field(description="Why this specific revision increases relevance for this role.")

class ResumeAnalysisReport(BaseModel):
    extracted_jobs: List[JobTarget] = Field(description="All job titles and companies identified directly from the job postings.")
    matching_skills: List[str] = Field(description="Key technical skills present in both the resume and live job postings.")
    missing_skills: List[str] = Field(description="Critical technical skills demanded by job market data but missing from resume.")
    skill_gap_summary: str = Field(description="High-level assessment of candidate readiness and alignment.")
    bullet_rewrites: List[ActionableRewrite] = Field(description="3 to 5 concrete resume bullet point improvements.")

# LCEL Pipeline Assembly & Execution
Concept: 
- LangChain Expression Language (LCEL) connects components into a pipeline using the Unix pipe operator (|).
- Here, prompt | structured_llm takes dictionary inputs, populates the system/user templates, forwards the structured query to Gemini, and automatically parses the LLM output into your validated ResumeAnalysisReport Pydantic object.

In [6]:
# 1. Define Prompt Template (target_role removed)
system_prompt = """You are a Senior Technical Recruiter and AI Career Coach specializing in Machine Learning, Data Science, and AI engineering roles.
Your task is to analyze a candidate's resume against local target job descriptions.

First, identify all job titles and company names present in the job descriptions.
Then, compare the candidate's resume against those requirements and generate a structured analysis report matching the JSON schema."""

user_prompt = f"""
CANDIDATE RESUME:
{resume_context}

TARGET JOB POSTINGS:
{job_data}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

# 2. Bind Structured Output
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.1)
structured_llm = llm.with_structured_output(ResumeAnalysisReport)

# 3. LCEL Chain Assembly
chain = prompt | structured_llm


In [7]:
%time

resume_path = 'data/resume.pdf'
job_desc_path = 'data/job_desc.txt'
output_log_file_path = 'data/resume_analysis_history.txt'

print(f"--> Parsing resume file ({resume_path})...")
resume_text = load_resume_text(resume_path)

print(f"--> Loading cleaned job descriptions from ({job_desc_path})...")
job_data = load_and_clean_job_descriptions(job_desc_path)

print("--> Invoking LCEL Structured Chain...")
report: ResumeAnalysisReport = chain.invoke({
    "resume_text": resume_text,
    "job_data": job_data
})

CPU times: user 1 μs, sys: 0 ns, total: 1 μs
Wall time: 2.15 μs
--> Parsing resume file (data/resume.pdf)...
--> Loading cleaned job descriptions from (data/job_desc.txt)...
[*] Successfully processed 2 job posting(s). Total characters: 10074
[*] Phase 2 Cleanup Complete. Total characters: 10062
--> Invoking LCEL Structured Chain...


In [20]:
print(report.model_dump_json(indent=2))

{
  "extracted_jobs": [
    {
      "company_name": "Oper",
      "job_title": "AI Solutions Engineer"
    },
    {
      "company_name": "Almedia",
      "job_title": "Data Scientist"
    }
  ],
  "matching_skills": [
    "Python",
    "SQL",
    "A/B Testing",
    "Machine Learning",
    "Statistical Modeling",
    "PyTorch",
    "LightGBM",
    "CatBoost",
    "Data Pipelines",
    "Causal Inference / Experimentation",
    "LLMs & NLP"
  ],
  "missing_skills": [
    "RAG Architectures (LangChain, LlamaIndex)",
    "Agent Frameworks (CrewAI, AutoGen, LangGraph)",
    "LLM Evaluation Harnesses (Ragas, TruLens)",
    "Production API Integration & Solutions Architecture",
    "Causal Reasoning / Uplift Modeling Tools (DoWhy, EconML)"
  ],
  "skill_gap_summary": "The candidate possesses a strong foundation in production data science (Afiniti) and deep learning research (Eckerlab, GippLab), aligning exceptionally well with Almedia's demand for predictive modeling, A/B testing, and rigorou

In [15]:
report.extracted_jobs

[JobTarget(company_name='Oper', job_title='AI Solutions Engineer'),
 JobTarget(company_name='Almedia', job_title='Data Scientist')]

In [21]:
print("\nExtracted Job Postings Identified:")
for job in report.extracted_jobs:
    print(f" • {job.company_name} - {job.job_title}")


Extracted Job Postings Identified:
 • Oper - AI Solutions Engineer
 • Almedia - Data Scientist


In [29]:
# Run number and time
run_id_with_time = f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

# Format list of company-jobtitle lines
job_targets_formatted = "\n".join([
    f"- {job.company_name} - {job.job_title}" for job in report.extracted_jobs
])

# Build full human-readable report string
report_text = f"""
Matching Skills:
{', '.join(report.matching_skills)}
------------------------------------------------
Missing Skills:
{', '.join(report.missing_skills)}
------------------------------------------------
Summary:
{report.skill_gap_summary}
------------------------------------------------
Bullet Rewrites:
"""

for idx, rewrite in enumerate(report.bullet_rewrites, 1):
    report_text += f"\n{idx}. Current: {rewrite.current_text}\n   Suggested: {rewrite.suggested_text}\n   Reasoning: {rewrite.reasoning}\n"

report_text = f"{run_id_with_time}\n{job_targets_formatted}\n\nFull ResumeReport text:\n{report_text.strip()}\n================================================\n\n"

print(report_text)

Time: 2026-10-17 11:18:48
- Oper - AI Solutions Engineer
- Almedia - Data Scientist

Full ResumeReport text:
Matching Skills:
Python, SQL, A/B Testing, Machine Learning, Statistical Modeling, PyTorch, LightGBM, CatBoost, Data Pipelines, Causal Inference / Experimentation, LLMs & NLP
------------------------------------------------
Missing Skills:
RAG Architectures (LangChain, LlamaIndex), Agent Frameworks (CrewAI, AutoGen, LangGraph), LLM Evaluation Harnesses (Ragas, TruLens), Production API Integration & Solutions Architecture, Causal Reasoning / Uplift Modeling Tools (DoWhy, EconML)
------------------------------------------------
Summary:
The candidate possesses a strong foundation in production data science (Afiniti) and deep learning research (Eckerlab, GippLab), aligning exceptionally well with Almedia's demand for predictive modeling, A/B testing, and rigorous analytical depth. For Oper's AI Solutions Engineer role, while the candidate lists 'Agentic AI' and LLM publications, th

In [ ]:
# Save output to disk
with open(output_log_file_path, "a", encoding="utf-8") as f:
        f.write(report_text)